# NBA One-Game Lineup Stint Pipeline (End-to-End)

This notebook reproduces the full pipeline for one game using player IDs as the source of truth.

1. Load cached `playbyplayv3` actions
2. Build grouped timestamps and grouped same-clock substitutions
3. Infer period-opening lineups (Q1 from live starters, Q2/Q3/Q4 inferred)
4. Validate inferred lineups
5. Build final `fact_lineup_stint`
6. Run quality checks

In [2]:
import sys
sys.path.append("/Users/manishkavuri/Desktop/nba-lineup-decision-engine")

In [3]:
import re
import pandas as pd

from src.ingestion.pbp_json import load_pbp_actions_dataframe
from src.processing.clock import parse_clock_seconds_remaining
from nba_api.live.nba.endpoints import boxscore as live_boxscore

pd.set_option("display.max_colwidth", 120)

In [4]:
# Config (same game used in prior work)
GAME_ID = "0042500111"

# Load and sort actions
pbp_df = load_pbp_actions_dataframe(GAME_ID).copy()
pbp_df = pbp_df.sort_values("actionNumber", kind="mergesort").reset_index(drop=True)
pbp_df["clock_seconds_remaining"] = pbp_df["clock"].apply(parse_clock_seconds_remaining)

print("Loaded actions:", len(pbp_df))
display(pbp_df[["actionNumber", "period", "clock", "teamId", "personId", "playerName", "actionType", "description", "scoreHome", "scoreAway"]].head(10))

Loaded actions: 485


,actionNumber,period,clock,teamId,personId,playerName,actionType,description,scoreHome,scoreAway
0,2,1,PT12M00.00S,0,0,,period,Start of 1st Period (1:11 PM EST),0,0
1,4,1,PT12M00.00S,1610612738,1629674,Queta,Jump Ball,Jump Ball Queta vs. Bona: Tip to Edgecombe,,
2,7,1,PT11M48.00S,1610612755,202331,George,Missed Shot,MISS George 26' 3PT Jump Shot,,
3,8,1,PT11M47.00S,1610612738,1628369,Tatum,Rebound,Tatum REBOUND (Off:0 Def:1),,
4,9,1,PT11M38.00S,1610612755,1641737,Bona,Foul,Bona S.FOUL (P1.T1) (S.Foster),,
5,11,1,PT11M38.00S,1610612738,1629674,Queta,Free Throw,Queta Free Throw 1 of 2 (1 PTS),1,0
6,12,1,PT11M38.00S,1610612738,1629674,Queta,Free Throw,MISS Queta Free Throw 2 of 2,,
7,13,1,PT11M38.00S,1610612755,1626162,Oubre Jr.,Rebound,Oubre Jr. REBOUND (Off:0 Def:1),,
8,14,1,PT11M24.00S,1610612755,1642845,Edgecombe,Missed Shot,MISS Edgecombe 3PT Jump Shot,,
9,15,1,PT11M23.00S,1610612738,1628369,Tatum,Rebound,Tatum REBOUND (Off:0 Def:2),,


In [5]:
# Player master from live boxscore (for IDs + readable names + Q1 starters)
live_bs = live_boxscore.BoxScore(game_id=GAME_ID).get_dict()["game"]
home_team_id = int(live_bs["homeTeam"]["teamId"])
away_team_id = int(live_bs["awayTeam"]["teamId"])

player_rows = []
for team_key, tid in [("homeTeam", home_team_id), ("awayTeam", away_team_id)]:
    for p in live_bs[team_key]["players"]:
        player_rows.append({
            "team_id": int(tid),
            "player_id": int(p["personId"]),
            "display_name": str(p.get("name") or p.get("familyName") or p.get("personId")),
            "starter": int(p.get("starter", 0)) == 1,
        })

player_master_df = pd.DataFrame(player_rows).drop_duplicates(subset=["team_id", "player_id"]).reset_index(drop=True)
player_id_to_name = dict(zip(player_master_df["player_id"], player_master_df["display_name"]))

def canon_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

normalized_name_to_player_id = {
    (int(r["team_id"]), canon_name(r["display_name"])): int(r["player_id"])
    for _, r in player_master_df.iterrows()
}

q1_home_starters = set(player_master_df[(player_master_df["team_id"] == home_team_id) & (player_master_df["starter"])]["player_id"].astype(int).tolist())
q1_away_starters = set(player_master_df[(player_master_df["team_id"] == away_team_id) & (player_master_df["starter"])]["player_id"].astype(int).tolist())

assert len(q1_home_starters) == 5 and len(q1_away_starters) == 5
print("Q1 HOME starters:", sorted(player_id_to_name.get(i, i) for i in q1_home_starters))
print("Q1 AWAY starters:", sorted(player_id_to_name.get(i, i) for i in q1_away_starters))

Q1 HOME starters: ['Derrick White', 'Jaylen Brown', 'Jayson Tatum', 'Neemias Queta', 'Sam Hauser']
Q1 AWAY starters: ['Adem Bona', 'Kelly Oubre Jr.', 'Paul George', 'Tyrese Maxey', 'VJ Edgecombe']


In [6]:
# Group same-clock substitution rows
sub_df = pbp_df[pbp_df["actionType"].eq("Substitution")].copy()
SUB_PATTERN = re.compile(r"^SUB:\\s*(?P<player_in>.+?)\\s+FOR\\s+(?P<player_out>.+?)\\s*$", re.IGNORECASE)
parsed = sub_df["description"].astype(str).str.extract(SUB_PATTERN)

sub_df["team_id"] = pd.to_numeric(sub_df["teamId"], errors="coerce").astype("Int64")
sub_df["player_out_id"] = pd.to_numeric(sub_df["personId"], errors="coerce").astype("Int64")
sub_df["player_in_name"] = parsed["player_in"].astype(str).str.strip()
sub_df["player_in_id"] = sub_df.apply(
    lambda r: normalized_name_to_player_id.get((int(r["team_id"]), canon_name(r["player_in_name"]))) if pd.notna(r["team_id"]) else None,
    axis=1,
)

sub_events_clean = sub_df[["actionNumber", "period", "clock", "team_id", "player_out_id", "player_in_id", "description"]].copy()
sub_events_clean["player_out_id"] = pd.to_numeric(sub_events_clean["player_out_id"], errors="coerce").astype("Int64")
sub_events_clean["player_in_id"] = pd.to_numeric(sub_events_clean["player_in_id"], errors="coerce").astype("Int64")

sub_groups_df = (
    sub_events_clean.groupby(["period", "clock"], sort=False)
    .apply(lambda g: pd.Series({"n_subs": len(g), "substitutions": g.sort_values("actionNumber").to_dict("records")}))
    .reset_index()
)

sub_group_lookup = {(int(r["period"]), r["clock"]): r["substitutions"] for _, r in sub_groups_df.iterrows()}

# Group all events by timestamp
event_groups_df = (
    pbp_df.groupby(["period", "clock", "clock_seconds_remaining"], sort=False)
    .agg(n_events=("actionNumber", "count"), action_numbers=("actionNumber", list), action_types=("actionType", list))
    .reset_index()
    .sort_values(["period", "clock_seconds_remaining"], ascending=[True, False])
    .reset_index(drop=True)
)

sub_keys = set(zip(sub_groups_df["period"].astype(int), sub_groups_df["clock"]))
event_groups_df["has_substitution"] = event_groups_df.apply(lambda r: (int(r["period"]), r["clock"]) in sub_keys, axis=1)

print("Sub batches:", len(sub_groups_df), "| Event groups:", len(event_groups_df))
display(sub_groups_df.head(8))

Sub batches: 31 | Event groups: 329


,period,clock,n_subs,substitutions
0,1,PT10M23.00S,1,"[{'actionNumber': 24, 'team_id': 1610612755, 'player_out_id': 1641737, 'player_in_id': None, 'description': 'SUB: Dr..."
1,1,PT08M39.00S,1,"[{'actionNumber': 49, 'team_id': 1610612738, 'player_out_id': 1629674, 'player_in_id': None, 'description': 'SUB: Vu..."
2,1,PT04M50.00S,3,"[{'actionNumber': 91, 'team_id': 1610612738, 'player_out_id': 1630573, 'player_in_id': None, 'description': 'SUB: Pr..."
3,1,PT04M39.00S,1,"[{'actionNumber': 106, 'team_id': 1610612755, 'player_out_id': 203083, 'player_in_id': None, 'description': 'SUB: Ou..."
4,1,PT03M49.00S,1,"[{'actionNumber': 120, 'team_id': 1610612738, 'player_out_id': 1627759, 'player_in_id': None, 'description': 'SUB: W..."
5,1,PT02M39.00S,1,"[{'actionNumber': 133, 'team_id': 1610612755, 'player_out_id': 202331, 'player_in_id': None, 'description': 'SUB: Ed..."
6,1,PT00M27.30S,2,"[{'actionNumber': 157, 'team_id': 1610612738, 'player_out_id': 202696, 'player_in_id': None, 'description': 'SUB: Br..."
7,2,PT08M43.00S,3,"[{'actionNumber': 221, 'team_id': 1610612738, 'player_out_id': 1630568, 'player_in_id': None, 'description': 'SUB: W..."


In [7]:
# Helpers for lineup inference and validation
def _to_int(x):
    try:
        return int(x)
    except Exception:
        return None

def ids_to_names(ids):
    return [player_id_to_name.get(int(i), f"id:{int(i)}") for i in sorted(set(int(x) for x in ids))]

def get_first_sub_roles_for_period(period):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q.empty:
        return {"home": {}, "away": {}, "home_out": set(), "home_in": set(), "away_out": set(), "away_in": set()}
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    q = q.sort_values("sec", ascending=False)

    home_roles, away_roles = {}, {}
    for _, row in q.iterrows():
        for sub in row["substitutions"]:
            team_id = _to_int(sub.get("team_id"))
            out_id = _to_int(sub.get("player_out_id"))
            in_id = _to_int(sub.get("player_in_id"))
            roles = home_roles if team_id == home_team_id else away_roles if team_id == away_team_id else None
            if roles is None:
                continue
            if out_id is not None and out_id not in roles:
                roles[out_id] = "OUT"
            if in_id is not None and in_id not in roles:
                roles[in_id] = "IN"

    return {
        "home": home_roles,
        "away": away_roles,
        "home_out": {pid for pid, role in home_roles.items() if role == "OUT"},
        "home_in": {pid for pid, role in home_roles.items() if role == "IN"},
        "away_out": {pid for pid, role in away_roles.items() if role == "OUT"},
        "away_in": {pid for pid, role in away_roles.items() if role == "IN"},
    }

def get_first_sub_clock(period):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q.empty:
        return None
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    return q.sort_values("sec", ascending=False).iloc[0]["clock"]

def get_players_seen_before_first_sub(period, first_sub_clock):
    q = pbp_df[pbp_df["period"].eq(period)].copy()
    if "clock_seconds_remaining" not in q.columns:
        q["clock_seconds_remaining"] = q["clock"].apply(parse_clock_seconds_remaining)
    fs = parse_clock_seconds_remaining(first_sub_clock) if first_sub_clock else None
    if fs is not None:
        q = q[(q["clock_seconds_remaining"] < 720.0) & (q["clock_seconds_remaining"] > fs)]
    q = q[
        (~q["actionType"].astype(str).str.lower().eq("period"))
        & (~q["actionType"].astype(str).str.lower().eq("substitution"))
    ].copy()
    q["personId_int"] = pd.to_numeric(q["personId"], errors="coerce")
    q["teamId_int"] = pd.to_numeric(q["teamId"], errors="coerce")
    q = q[(q["personId_int"].notna()) & (q["personId_int"] != 0) & (q["teamId_int"].isin([home_team_id, away_team_id]))]
    home_seen = set(q[q["teamId_int"].eq(home_team_id)]["personId_int"].astype(int).tolist())
    away_seen = set(q[q["teamId_int"].eq(away_team_id)]["personId_int"].astype(int).tolist())
    return home_seen, away_seen

def apply_sub_batch_ids(home_ids, away_ids, sub_batch):
    home, away = set(home_ids), set(away_ids)
    debug = []
    for sub in sub_batch or []:
        team_id = _to_int(sub.get("team_id"))
        out_id = _to_int(sub.get("player_out_id"))
        in_id = _to_int(sub.get("player_in_id"))
        lineup = home if team_id == home_team_id else away if team_id == away_team_id else None
        if lineup is None:
            continue
        ok = (out_id in lineup) and (in_id not in lineup) and (in_id is not None)
        if ok:
            lineup.remove(out_id)
            lineup.add(in_id)
        debug.append({"team_id": team_id, "out_id": out_id, "in_id": in_id, "applied": ok})
    return home, away, debug

In [8]:
# Infer period_start_lineups_inferred (Q1 official + Q2/Q3/Q4 inferred)
period_start_lineups_inferred = {1: {"home": set(q1_home_starters), "away": set(q1_away_starters)}}

def period_end_lineups_from_openers(period, home_open, away_open):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    if q.empty:
        return set(home_open), set(away_open)
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    q = q.sort_values("sec", ascending=False)
    h, a = set(home_open), set(away_open)
    for _, row in q.iterrows():
        h, a, _ = apply_sub_batch_ids(h, a, sub_group_lookup.get((int(row["period"]), row["clock"]), []))
    return h, a

prev_home_end, prev_away_end = period_end_lineups_from_openers(1, q1_home_starters, q1_away_starters)
debug_rows = []

for period in [2, 3, 4]:
    roles = get_first_sub_roles_for_period(period)
    first_sub_clock = get_first_sub_clock(period)
    seen_home, seen_away = get_players_seen_before_first_sub(period, first_sub_clock)

    fallback_home = set(q1_home_starters if period == 3 else prev_home_end)
    fallback_away = set(q1_away_starters if period == 3 else prev_away_end)

    home_candidates = list(sorted(roles["home_out"])) + list(sorted(seen_home - roles["home_in"])) + list(sorted(fallback_home - roles["home_in"]))
    away_candidates = list(sorted(roles["away_out"])) + list(sorted(seen_away - roles["away_in"])) + list(sorted(fallback_away - roles["away_in"]))

    inferred_home, inferred_away = [], []
    for pid in home_candidates:
        if pid not in inferred_home:
            inferred_home.append(pid)
        if len(inferred_home) == 5:
            break
    for pid in away_candidates:
        if pid not in inferred_away:
            inferred_away.append(pid)
        if len(inferred_away) == 5:
            break

    period_start_lineups_inferred[period] = {"home": set(inferred_home), "away": set(inferred_away)}
    debug_rows.append({
        "period": period,
        "first_sub_clock": first_sub_clock,
        "home_out": ids_to_names(roles["home_out"]),
        "home_in": ids_to_names(roles["home_in"]),
        "home_seen": ids_to_names(seen_home),
        "home_final": ids_to_names(inferred_home),
        "away_out": ids_to_names(roles["away_out"]),
        "away_in": ids_to_names(roles["away_in"]),
        "away_seen": ids_to_names(seen_away),
        "away_final": ids_to_names(inferred_away),
    })

    prev_home_end, prev_away_end = period_end_lineups_from_openers(period, inferred_home, inferred_away)

display(pd.DataFrame(debug_rows))

,period,first_sub_clock,home_out,home_in,home_seen,home_final,away_out,away_in,away_seen,away_final
0,2,PT08M43.00S,"[Nikola Vučević, Jaylen Brown, Neemias Queta, Payton Pritchard, Luka Garza, Sam Hauser, Baylor Scheierman]",[],"[Jaylen Brown, Payton Pritchard, Luka Garza, Sam Hauser, Baylor Scheierman]","[Nikola Vučević, Jaylen Brown, Neemias Queta, Payton Pritchard, Luka Garza]","[Paul George, Andre Drummond, Quentin Grimes, Adem Bona, Justin Edwards]",[],"[Paul George, Andre Drummond, Quentin Grimes, Justin Edwards, VJ Edgecombe]","[Paul George, Andre Drummond, Quentin Grimes, Adem Bona, Justin Edwards]"
1,3,PT10M25.00S,"[Nikola Vučević, Jayson Tatum, Neemias Queta, Luka Garza, Sam Hauser]",[],"[Derrick White, Neemias Queta]","[Nikola Vučević, Jayson Tatum, Neemias Queta, Luka Garza, Sam Hauser]","[Paul George, Andre Drummond, Kelly Oubre Jr., Adem Bona, VJ Edgecombe]",[],"[Paul George, Andre Drummond, Kelly Oubre Jr., Tyrese Maxey]","[Paul George, Andre Drummond, Kelly Oubre Jr., Adem Bona, VJ Edgecombe]"
2,4,PT09M16.00S,"[Jayson Tatum, Neemias Queta, Payton Pritchard, Sam Hauser]",[],"[Jayson Tatum, Neemias Queta, Payton Pritchard, Sam Hauser, Baylor Scheierman]","[Jayson Tatum, Neemias Queta, Payton Pritchard, Sam Hauser, Baylor Scheierman]","[Andre Drummond, Kelly Oubre Jr., Tyrese Maxey, Dominick Barlow, Adem Bona, Justin Edwards, VJ Edgecombe]",[],"[Quentin Grimes, Dominick Barlow, Justin Edwards, VJ Edgecombe]","[Andre Drummond, Kelly Oubre Jr., Tyrese Maxey, Dominick Barlow, Adem Bona]"


In [9]:
# Strong validation: first 3 sub batches in each period must apply cleanly
def validate_first_n_sub_batches(period, n_batches=3):
    q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()
    q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
    q = q.sort_values("sec", ascending=False).head(n_batches)
    h = set(period_start_lineups_inferred[period]["home"])
    a = set(period_start_lineups_inferred[period]["away"])
    rows = []
    for _, row in q.iterrows():
        for sub in sub_group_lookup.get((int(row["period"]), row["clock"]), []):
            team_id = _to_int(sub.get("team_id"))
            out_id = _to_int(sub.get("player_out_id"))
            in_id = _to_int(sub.get("player_in_id"))
            lineup = h if team_id == home_team_id else a if team_id == away_team_id else None
            if lineup is None:
                continue
            out_found = out_id in lineup
            in_already = in_id in lineup if in_id is not None else True
            ok = out_found and (not in_already) and (in_id is not None)
            if ok:
                lineup.remove(out_id)
                lineup.add(in_id)
            rows.append({
                "period": period,
                "clock": row["clock"],
                "outgoing_found": out_found,
                "incoming_already_present": in_already,
                "size_ok": len(h) == 5 and len(a) == 5,
                "applied": ok,
            })
    out = pd.DataFrame(rows)
    out["pass"] = out["outgoing_found"] & (~out["incoming_already_present"]) & out["size_ok"]
    return out

validation_df = pd.concat([validate_first_n_sub_batches(p, 3) for p in [2, 3, 4]], ignore_index=True)
print("Validation pass rows:", int(validation_df["pass"].sum()), "/", len(validation_df))
display(validation_df)

Validation pass rows: 0 / 14


,period,clock,outgoing_found,incoming_already_present,size_ok,applied,pass
0,2,PT08M43.00S,True,True,True,False,False
1,2,PT08M43.00S,False,True,True,False,False
2,2,PT08M43.00S,True,True,True,False,False
3,2,PT07M16.00S,False,True,True,False,False
4,2,PT07M16.00S,True,True,True,False,False
5,2,PT06M50.00S,True,True,True,False,False
6,3,PT10M25.00S,True,True,True,False,False
7,3,PT06M45.00S,True,True,True,False,False
8,3,PT04M33.00S,True,True,True,False,False
9,3,PT04M33.00S,True,True,True,False,False


In [10]:
# Build final fact_lineup_stint
def score_snapshot(period, clock, last_known=(0, 0)):
    rows = pbp_df[(pbp_df["period"].eq(period)) & (pbp_df["clock"].eq(clock))].copy()
    if rows.empty:
        return last_known
    sh = pd.to_numeric(rows["scoreHome"], errors="coerce").dropna()
    sa = pd.to_numeric(rows["scoreAway"], errors="coerce").dropna()
    return (int(sh.iloc[-1]) if len(sh) else int(last_known[0]), int(sa.iloc[-1]) if len(sa) else int(last_known[1]))

def build_row(period, start_clock, end_clock, home_ids, away_ids, sh0, sa0, sh1, sa1, ended_by_sub):
    s0 = float(parse_clock_seconds_remaining(start_clock))
    s1 = float(parse_clock_seconds_remaining(end_clock))
    home_ids = sorted(int(i) for i in home_ids)
    away_ids = sorted(int(i) for i in away_ids)
    return {
        "game_id": GAME_ID,
        "period": int(period),
        "start_clock": start_clock,
        "end_clock": end_clock,
        "start_seconds_remaining": s0,
        "end_seconds_remaining": s1,
        "duration_seconds": s0 - s1,
        "home_team_id": home_team_id,
        "away_team_id": away_team_id,
        "home_lineup_ids": home_ids,
        "away_lineup_ids": away_ids,
        "home_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in home_ids],
        "away_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in away_ids],
        "score_home_start": int(sh0),
        "score_away_start": int(sa0),
        "score_home_end": int(sh1),
        "score_away_end": int(sa1),
        "ended_by_substitution": bool(ended_by_sub),
    }

rows = []
last_known_score = (0, 0)
for period in [1, 2, 3, 4]:
    home_state = set(period_start_lineups_inferred[period]["home"])
    away_state = set(period_start_lineups_inferred[period]["away"])
    start_clock = "PT12M00.00S"
    sh_start, sa_start = score_snapshot(period, start_clock, last_known_score)
    cur_start_clock = start_clock
    cur_sh, cur_sa = sh_start, sa_start

    period_groups = event_groups_df[event_groups_df["period"].eq(period)].sort_values("clock_seconds_remaining", ascending=False)
    for _, g in period_groups.iterrows():
        clock = g["clock"]
        sh_here, sa_here = score_snapshot(period, clock, last_known_score)
        last_known_score = (sh_here, sa_here)
        if not bool(g["has_substitution"]):
            continue
        rows.append(build_row(period, cur_start_clock, clock, home_state, away_state, cur_sh, cur_sa, sh_here, sa_here, True))
        home_state, away_state, _ = apply_sub_batch_ids(home_state, away_state, sub_group_lookup.get((period, clock), []))
        cur_start_clock = clock
        cur_sh, cur_sa = sh_here, sa_here

    end_clock = "PT00M00.00S"
    sh_end, sa_end = score_snapshot(period, end_clock, last_known_score)
    last_known_score = (sh_end, sa_end)
    rows.append(build_row(period, cur_start_clock, end_clock, home_state, away_state, cur_sh, cur_sa, sh_end, sa_end, False))

fact_lineup_stint = pd.DataFrame(rows)
print("fact_lineup_stint rows:", len(fact_lineup_stint))
display(fact_lineup_stint.head(20))

fact_lineup_stint rows: 35


,game_id,period,start_clock,end_clock,start_seconds_remaining,end_seconds_remaining,duration_seconds,home_team_id,away_team_id,home_lineup_ids,away_lineup_ids,home_lineup_names,away_lineup_names,score_home_start,score_away_start,score_home_end,score_away_end,ended_by_substitution
0,0042500111,1,PT12M00.00S,PT10M23.00S,720.0,623.0,97.0,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",0,0,4,3,True
1,0042500111,1,PT10M23.00S,PT08M39.00S,623.0,519.0,104.0,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",4,3,8,7,True
2,0042500111,1,PT08M39.00S,PT04M50.00S,519.0,290.0,229.0,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",8,7,22,11,True
3,0042500111,1,PT04M50.00S,PT04M39.00S,290.0,279.0,11.0,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",22,11,22,11,True
4,0042500111,1,PT04M39.00S,PT03M49.00S,279.0,229.0,50.0,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",22,11,22,14,True
5,0042500111,1,PT03M49.00S,PT02M39.00S,229.0,159.0,70.0,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",22,14,24,16,True
6,0042500111,1,PT02M39.00S,PT00M27.30S,159.0,27.3,131.7,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",24,16,31,18,True
7,0042500111,1,PT00M27.30S,PT00M00.00S,27.3,0.0,27.3,1610612738,1610612755,"[1627759, 1628369, 1628401, 1629674, 1630573]","[202331, 1626162, 1630178, 1641737, 1642845]","[Jaylen Brown, Jayson Tatum, Derrick White, Neemias Queta, Sam Hauser]","[Paul George, Kelly Oubre Jr., Tyrese Maxey, Adem Bona, VJ Edgecombe]",31,18,33,18,False
8,0042500111,2,PT12M00.00S,PT08M43.00S,720.0,523.0,197.0,1610612738,1610612755,"[202696, 1627759, 1629674, 1630202, 1630568]","[202331, 203083, 1629656, 1641737, 1642348]","[Nikola Vučević, Jaylen Brown, Neemias Queta, Payton Pritchard, Luka Garza]","[Paul George, Andre Drummond, Quentin Grimes, Adem Bona, Justin Edwards]",33,18,42,27,True
9,0042500111,2,PT08M43.00S,PT07M16.00S,523.0,436.0,87.0,1610612738,1610612755,"[202696, 1627759, 1629674, 1630202, 1630568]","[202331, 203083, 1629656, 1641737, 1642348]","[Nikola Vučević, Jaylen Brown, Neemias Queta, Payton Pritchard, Luka Garza]","[Paul George, Andre Drummond, Quentin Grimes, Adem Bona, Justin Edwards]",42,27,49,29,True


In [11]:
# Quality checks
neg_dur = int((fact_lineup_stint["duration_seconds"] < 0).sum())
zero_dur = int((fact_lineup_stint["duration_seconds"] == 0).sum())
bad_home = int(fact_lineup_stint["home_lineup_ids"].apply(lambda x: len(set(x)) != 5).sum())
bad_away = int(fact_lineup_stint["away_lineup_ids"].apply(lambda x: len(set(x)) != 5).sum())
total_sec = float(fact_lineup_stint["duration_seconds"].sum())

# Lineup changes should only happen at substitution boundaries
f = fact_lineup_stint.sort_values(["period", "start_seconds_remaining"], ascending=[True, False]).reset_index(drop=True)
boundary_issues = 0
for i in range(len(f) - 1):
    a, b = f.iloc[i], f.iloc[i + 1]
    if int(a["period"]) != int(b["period"]):
        continue
    lineup_changed = (set(a["home_lineup_ids"]) != set(b["home_lineup_ids"])) or (set(a["away_lineup_ids"]) != set(b["away_lineup_ids"]))
    if lineup_changed and (not bool(a["ended_by_substitution"]) or str(a["end_clock"]) != str(b["start_clock"])):
        boundary_issues += 1

print("Negative durations:", neg_dur)
print("Zero durations:", zero_dur)
print("Bad home lineup size:", bad_home)
print("Bad away lineup size:", bad_away)
print("Total regulation seconds:", total_sec, "(target=2880)")
print("Boundary sanity issues:", boundary_issues)

Negative durations: 0
Zero durations: 0
Bad home lineup size: 0
Bad away lineup size: 0
Total regulation seconds: 2880.0 (target=2880)
Boundary sanity issues: 0


In [14]:
import json
import time
from pathlib import Path

import pandas as pd
from nba_api.live.nba.endpoints import playbyplay as live_playbyplay


# Make sure this is your actual project root
PROJECT_ROOT = "/Users/manishkavuri/Desktop/nba-lineup-decision-engine"

RAW_PBP_DIR = Path(PROJECT_ROOT) / "data/raw/pbp"
RAW_PBP_DIR.mkdir(parents=True, exist_ok=True)


def fetch_pbp_json_from_nba(game_id):
    """
    Downloads raw play-by-play JSON from nba_api live endpoint.
    """
    game_id = str(game_id).zfill(10)

    pbp = live_playbyplay.PlayByPlay(game_id=game_id)
    raw = pbp.get_dict()

    return raw


def load_pbp_actions_dataframe(game_id, force_refresh=False):
    """
    Cache-first loader:
    1. If data/raw/pbp/{game_id}.json exists, read it.
    2. If missing, download from NBA API and save locally.
    3. Return the actions as a pandas DataFrame.
    """
    game_id = str(game_id).zfill(10)
    local_path = RAW_PBP_DIR / f"{game_id}.json"

    if local_path.exists() and not force_refresh:
        with open(local_path, "r") as f:
            raw = json.load(f)
    else:
        print(f"Downloading play-by-play for {game_id}...")
        raw = fetch_pbp_json_from_nba(game_id)

        with open(local_path, "w") as f:
            json.dump(raw, f)

        print(f"Saved raw PBP to {local_path}")

    actions = raw.get("game", {}).get("actions", [])

    if not actions:
        raise ValueError(f"No play-by-play actions found for game {game_id}")

    df = pd.DataFrame(actions)

    # Ensure expected columns exist
    expected_cols = [
        "actionNumber",
        "period",
        "clock",
        "actionType",
        "description",
        "teamId",
        "personId",
        "scoreHome",
        "scoreAway",
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = None

    return df

In [15]:
# If needed, run once:
# !pip install nba_api boto3 pyarrow

import sys
import re
import time
import traceback
from pathlib import Path

import pandas as pd
import boto3

from nba_api.stats.endpoints import leaguegamefinder
from nba_api.live.nba.endpoints import boxscore as live_boxscore

# Update this path if needed
PROJECT_ROOT = "/Users/manishkavuri/Desktop/nba-lineup-decision-engine"
sys.path.append(PROJECT_ROOT)

from src.ingestion.pbp_json import load_pbp_actions_dataframe
from src.processing.clock import parse_clock_seconds_remaining


# -----------------------------
# Config
# -----------------------------
SEASON = "2024-25"
SEASON_TYPE = "Regular Season"
N_GAMES = 5

LOCAL_OUTPUT_DIR = Path("data/processed/test_5_games_lineup_stints")

S3_UPLOAD = False   # change to True after local test works
S3_BUCKET = "your-s3-bucket-name"  # replace with your real bucket name
S3_PREFIX = "nba/processed/lineup_stints"


# -----------------------------
# Helpers
# -----------------------------
def canon_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def _to_int(x):
    try:
        if pd.isna(x):
            return None
        return int(x)
    except Exception:
        return None


def get_period_start_clock(period):
    """
    NBA regulation periods are 12 minutes.
    Overtime periods are 5 minutes.
    """
    if int(period) <= 4:
        return "PT12M00.00S"
    return "PT05M00.00S"


def get_period_start_seconds(period):
    return 720.0 if int(period) <= 4 else 300.0


def get_season_game_ids(season=SEASON, season_type=SEASON_TYPE, n_games=5):
    finder = leaguegamefinder.LeagueGameFinder(
        season_nullable=season,
        season_type_nullable=season_type
    )

    games = finder.get_data_frames()[0].copy()

    # GAME_ID appears once per team, so drop duplicates
    games["GAME_ID"] = games["GAME_ID"].astype(str).str.zfill(10)
    games = games.sort_values("GAME_DATE")

    game_ids = games["GAME_ID"].drop_duplicates().head(n_games).tolist()

    return game_ids


def save_df_local_and_s3(df, local_path, s3_bucket=None, s3_key=None, upload=False):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_parquet(local_path, index=False)

    if upload:
        if not s3_bucket or not s3_key:
            raise ValueError("s3_bucket and s3_key are required when upload=True")

        s3 = boto3.client("s3")
        s3.upload_file(
            Filename=str(local_path),
            Bucket=s3_bucket,
            Key=s3_key
        )

        return f"s3://{s3_bucket}/{s3_key}"

    return str(local_path)


# -----------------------------
# Main single-game function
# -----------------------------
def build_lineup_stints_for_game(game_id):
    game_id = str(game_id).zfill(10)

    # Load play-by-play
    pbp_df = load_pbp_actions_dataframe(game_id).copy()
    pbp_df = pbp_df.sort_values("actionNumber", kind="mergesort").reset_index(drop=True)
    pbp_df["clock_seconds_remaining"] = pbp_df["clock"].apply(parse_clock_seconds_remaining)

    # Load live boxscore
    live_bs = live_boxscore.BoxScore(game_id=game_id).get_dict()["game"]

    home_team_id = int(live_bs["homeTeam"]["teamId"])
    away_team_id = int(live_bs["awayTeam"]["teamId"])

    # Player master
    player_rows = []

    for team_key, tid in [("homeTeam", home_team_id), ("awayTeam", away_team_id)]:
        for p in live_bs[team_key]["players"]:
            player_rows.append({
                "team_id": int(tid),
                "player_id": int(p["personId"]),
                "display_name": str(p.get("name") or p.get("familyName") or p.get("personId")),
                "starter": int(p.get("starter", 0)) == 1,
            })

    player_master_df = (
        pd.DataFrame(player_rows)
        .drop_duplicates(subset=["team_id", "player_id"])
        .reset_index(drop=True)
    )

    player_id_to_name = dict(zip(player_master_df["player_id"], player_master_df["display_name"]))

    normalized_name_to_player_id = {
        (int(r["team_id"]), canon_name(r["display_name"])): int(r["player_id"])
        for _, r in player_master_df.iterrows()
    }

    q1_home_starters = set(
        player_master_df[
            (player_master_df["team_id"] == home_team_id)
            & (player_master_df["starter"])
        ]["player_id"].astype(int).tolist()
    )

    q1_away_starters = set(
        player_master_df[
            (player_master_df["team_id"] == away_team_id)
            & (player_master_df["starter"])
        ]["player_id"].astype(int).tolist()
    )

    if len(q1_home_starters) != 5 or len(q1_away_starters) != 5:
        raise ValueError(
            f"Starter issue for game {game_id}: "
            f"home={len(q1_home_starters)}, away={len(q1_away_starters)}"
        )

    # Parse substitutions
    sub_df = pbp_df[pbp_df["actionType"].eq("Substitution")].copy()

    SUB_PATTERN = re.compile(
        r"^SUB:\s*(?P<player_in>.+?)\s+FOR\s+(?P<player_out>.+?)\s*$",
        re.IGNORECASE
    )

    parsed = sub_df["description"].astype(str).str.extract(SUB_PATTERN)

    sub_df["team_id"] = pd.to_numeric(sub_df["teamId"], errors="coerce").astype("Int64")
    sub_df["player_out_id"] = pd.to_numeric(sub_df["personId"], errors="coerce").astype("Int64")
    sub_df["player_in_name"] = parsed["player_in"].astype(str).str.strip()

    sub_df["player_in_id"] = sub_df.apply(
        lambda r: normalized_name_to_player_id.get(
            (int(r["team_id"]), canon_name(r["player_in_name"]))
        ) if pd.notna(r["team_id"]) else None,
        axis=1,
    )

    sub_events_clean = sub_df[
        ["actionNumber", "period", "clock", "team_id", "player_out_id", "player_in_id", "description"]
    ].copy()

    sub_events_clean["player_out_id"] = pd.to_numeric(
        sub_events_clean["player_out_id"], errors="coerce"
    ).astype("Int64")

    sub_events_clean["player_in_id"] = pd.to_numeric(
        sub_events_clean["player_in_id"], errors="coerce"
    ).astype("Int64")

    # Group same-clock substitution rows
    sub_groups = []

    for (period, clock), g in sub_events_clean.groupby(["period", "clock"], sort=False):
        sub_groups.append({
            "period": int(period),
            "clock": clock,
            "n_subs": len(g),
            "substitutions": g.sort_values("actionNumber").to_dict("records")
        })

    sub_groups_df = pd.DataFrame(sub_groups)

    if sub_groups_df.empty:
        raise ValueError(f"No substitution groups found for game {game_id}")

    sub_group_lookup = {
        (int(r["period"]), r["clock"]): r["substitutions"]
        for _, r in sub_groups_df.iterrows()
    }

    # Group all events by timestamp
    event_groups_df = (
        pbp_df.groupby(["period", "clock", "clock_seconds_remaining"], sort=False)
        .agg(
            n_events=("actionNumber", "count"),
            action_numbers=("actionNumber", list),
            action_types=("actionType", list)
        )
        .reset_index()
        .sort_values(["period", "clock_seconds_remaining"], ascending=[True, False])
        .reset_index(drop=True)
    )

    sub_keys = set(zip(sub_groups_df["period"].astype(int), sub_groups_df["clock"]))

    event_groups_df["has_substitution"] = event_groups_df.apply(
        lambda r: (int(r["period"]), r["clock"]) in sub_keys,
        axis=1
    )

    # Helper functions that depend on this game's variables
    def ids_to_names(ids):
        return [
            player_id_to_name.get(int(i), f"id:{int(i)}")
            for i in sorted(set(int(x) for x in ids))
        ]

    def apply_sub_batch_ids(home_ids, away_ids, sub_batch):
        home, away = set(home_ids), set(away_ids)
        debug = []

        for sub in sub_batch or []:
            team_id = _to_int(sub.get("team_id"))
            out_id = _to_int(sub.get("player_out_id"))
            in_id = _to_int(sub.get("player_in_id"))

            lineup = home if team_id == home_team_id else away if team_id == away_team_id else None

            if lineup is None:
                continue

            ok = (out_id in lineup) and (in_id not in lineup) and (in_id is not None)

            if ok:
                lineup.remove(out_id)
                lineup.add(in_id)

            debug.append({
                "team_id": team_id,
                "out_id": out_id,
                "in_id": in_id,
                "applied": ok
            })

        return home, away, debug

    def get_first_sub_roles_for_period(period):
        q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()

        if q.empty:
            return {
                "home": {},
                "away": {},
                "home_out": set(),
                "home_in": set(),
                "away_out": set(),
                "away_in": set()
            }

        q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
        q = q.sort_values("sec", ascending=False)

        home_roles, away_roles = {}, {}

        for _, row in q.iterrows():
            for sub in row["substitutions"]:
                team_id = _to_int(sub.get("team_id"))
                out_id = _to_int(sub.get("player_out_id"))
                in_id = _to_int(sub.get("player_in_id"))

                roles = home_roles if team_id == home_team_id else away_roles if team_id == away_team_id else None

                if roles is None:
                    continue

                if out_id is not None and out_id not in roles:
                    roles[out_id] = "OUT"

                if in_id is not None and in_id not in roles:
                    roles[in_id] = "IN"

        return {
            "home": home_roles,
            "away": away_roles,
            "home_out": {pid for pid, role in home_roles.items() if role == "OUT"},
            "home_in": {pid for pid, role in home_roles.items() if role == "IN"},
            "away_out": {pid for pid, role in away_roles.items() if role == "OUT"},
            "away_in": {pid for pid, role in away_roles.items() if role == "IN"},
        }

    def get_first_sub_clock(period):
        q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()

        if q.empty:
            return None

        q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)

        return q.sort_values("sec", ascending=False).iloc[0]["clock"]

    def get_players_seen_before_first_sub(period, first_sub_clock):
        q = pbp_df[pbp_df["period"].eq(period)].copy()

        if "clock_seconds_remaining" not in q.columns:
            q["clock_seconds_remaining"] = q["clock"].apply(parse_clock_seconds_remaining)

        fs = parse_clock_seconds_remaining(first_sub_clock) if first_sub_clock else None
        period_start_sec = get_period_start_seconds(period)

        if fs is not None:
            q = q[
                (q["clock_seconds_remaining"] < period_start_sec)
                & (q["clock_seconds_remaining"] > fs)
            ]

        q = q[
            (~q["actionType"].astype(str).str.lower().eq("period"))
            & (~q["actionType"].astype(str).str.lower().eq("substitution"))
        ].copy()

        q["personId_int"] = pd.to_numeric(q["personId"], errors="coerce")
        q["teamId_int"] = pd.to_numeric(q["teamId"], errors="coerce")

        q = q[
            (q["personId_int"].notna())
            & (q["personId_int"] != 0)
            & (q["teamId_int"].isin([home_team_id, away_team_id]))
        ]

        home_seen = set(
            q[q["teamId_int"].eq(home_team_id)]["personId_int"].astype(int).tolist()
        )

        away_seen = set(
            q[q["teamId_int"].eq(away_team_id)]["personId_int"].astype(int).tolist()
        )

        return home_seen, away_seen

    def period_end_lineups_from_openers(period, home_open, away_open):
        q = sub_groups_df[sub_groups_df["period"].eq(period)].copy()

        if q.empty:
            return set(home_open), set(away_open)

        q["sec"] = q["clock"].apply(parse_clock_seconds_remaining)
        q = q.sort_values("sec", ascending=False)

        h, a = set(home_open), set(away_open)

        for _, row in q.iterrows():
            h, a, _ = apply_sub_batch_ids(
                h,
                a,
                sub_group_lookup.get((int(row["period"]), row["clock"]), [])
            )

        return h, a

    # Infer period starting lineups
    periods = sorted(pbp_df["period"].dropna().astype(int).unique().tolist())

    period_start_lineups_inferred = {
        1: {
            "home": set(q1_home_starters),
            "away": set(q1_away_starters)
        }
    }

    prev_home_end, prev_away_end = period_end_lineups_from_openers(
        1,
        q1_home_starters,
        q1_away_starters
    )

    debug_rows = []

    for period in periods:
        if period == 1:
            continue

        roles = get_first_sub_roles_for_period(period)
        first_sub_clock = get_first_sub_clock(period)
        seen_home, seen_away = get_players_seen_before_first_sub(period, first_sub_clock)

        # Your original notebook logic:
        # Q3 often resets to starters after halftime.
        # Other periods fall back to previous period ending lineup.
        fallback_home = set(q1_home_starters if period == 3 else prev_home_end)
        fallback_away = set(q1_away_starters if period == 3 else prev_away_end)

        home_candidates = (
            list(sorted(roles["home_out"]))
            + list(sorted(seen_home - roles["home_in"]))
            + list(sorted(fallback_home - roles["home_in"]))
        )

        away_candidates = (
            list(sorted(roles["away_out"]))
            + list(sorted(seen_away - roles["away_in"]))
            + list(sorted(fallback_away - roles["away_in"]))
        )

        inferred_home, inferred_away = [], []

        for pid in home_candidates:
            if pid not in inferred_home:
                inferred_home.append(pid)
            if len(inferred_home) == 5:
                break

        for pid in away_candidates:
            if pid not in inferred_away:
                inferred_away.append(pid)
            if len(inferred_away) == 5:
                break

        period_start_lineups_inferred[period] = {
            "home": set(inferred_home),
            "away": set(inferred_away)
        }

        debug_rows.append({
            "game_id": game_id,
            "period": period,
            "first_sub_clock": first_sub_clock,
            "home_final": ids_to_names(inferred_home),
            "away_final": ids_to_names(inferred_away),
            "home_size": len(inferred_home),
            "away_size": len(inferred_away)
        })

        prev_home_end, prev_away_end = period_end_lineups_from_openers(
            period,
            inferred_home,
            inferred_away
        )

    lineup_debug_df = pd.DataFrame(debug_rows)

    # Build fact_lineup_stint
    def score_snapshot(period, clock, last_known=(0, 0)):
        rows = pbp_df[
            (pbp_df["period"].eq(period))
            & (pbp_df["clock"].eq(clock))
        ].copy()

        if rows.empty:
            return last_known

        sh = pd.to_numeric(rows["scoreHome"], errors="coerce").dropna()
        sa = pd.to_numeric(rows["scoreAway"], errors="coerce").dropna()

        return (
            int(sh.iloc[-1]) if len(sh) else int(last_known[0]),
            int(sa.iloc[-1]) if len(sa) else int(last_known[1])
        )

    def build_row(
        period,
        start_clock,
        end_clock,
        home_ids,
        away_ids,
        sh0,
        sa0,
        sh1,
        sa1,
        ended_by_sub
    ):
        s0 = float(parse_clock_seconds_remaining(start_clock))
        s1 = float(parse_clock_seconds_remaining(end_clock))

        home_ids = sorted(int(i) for i in home_ids)
        away_ids = sorted(int(i) for i in away_ids)

        return {
            "game_id": game_id,
            "period": int(period),
            "start_clock": start_clock,
            "end_clock": end_clock,
            "start_seconds_remaining": s0,
            "end_seconds_remaining": s1,
            "duration_seconds": s0 - s1,
            "home_team_id": home_team_id,
            "away_team_id": away_team_id,
            "home_lineup_ids": home_ids,
            "away_lineup_ids": away_ids,
            "home_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in home_ids],
            "away_lineup_names": [player_id_to_name.get(i, f"id:{i}") for i in away_ids],
            "score_home_start": int(sh0),
            "score_away_start": int(sa0),
            "score_home_end": int(sh1),
            "score_away_end": int(sa1),
            "ended_by_substitution": bool(ended_by_sub),
        }

    rows = []
    last_known_score = (0, 0)

    for period in periods:
        if period not in period_start_lineups_inferred:
            continue

        home_state = set(period_start_lineups_inferred[period]["home"])
        away_state = set(period_start_lineups_inferred[period]["away"])

        start_clock = get_period_start_clock(period)

        sh_start, sa_start = score_snapshot(period, start_clock, last_known_score)

        cur_start_clock = start_clock
        cur_sh, cur_sa = sh_start, sa_start

        period_groups = (
            event_groups_df[event_groups_df["period"].eq(period)]
            .sort_values("clock_seconds_remaining", ascending=False)
        )

        for _, g in period_groups.iterrows():
            clock = g["clock"]
            sh_here, sa_here = score_snapshot(period, clock, last_known_score)
            last_known_score = (sh_here, sa_here)

            if not bool(g["has_substitution"]):
                continue

            rows.append(
                build_row(
                    period,
                    cur_start_clock,
                    clock,
                    home_state,
                    away_state,
                    cur_sh,
                    cur_sa,
                    sh_here,
                    sa_here,
                    True
                )
            )

            home_state, away_state, _ = apply_sub_batch_ids(
                home_state,
                away_state,
                sub_group_lookup.get((int(period), clock), [])
            )

            cur_start_clock = clock
            cur_sh, cur_sa = sh_here, sa_here

        end_clock = "PT00M00.00S"

        sh_end, sa_end = score_snapshot(period, end_clock, last_known_score)
        last_known_score = (sh_end, sa_end)

        rows.append(
            build_row(
                period,
                cur_start_clock,
                end_clock,
                home_state,
                away_state,
                cur_sh,
                cur_sa,
                sh_end,
                sa_end,
                False
            )
        )

    fact_lineup_stint = pd.DataFrame(rows)

    # Quality checks
    if fact_lineup_stint.empty:
        raise ValueError(f"No stint rows produced for game {game_id}")

    neg_dur = int((fact_lineup_stint["duration_seconds"] < 0).sum())
    zero_dur = int((fact_lineup_stint["duration_seconds"] == 0).sum())

    bad_home = int(
        fact_lineup_stint["home_lineup_ids"]
        .apply(lambda x: len(set(x)) != 5)
        .sum()
    )

    bad_away = int(
        fact_lineup_stint["away_lineup_ids"]
        .apply(lambda x: len(set(x)) != 5)
        .sum()
    )

    total_sec = float(fact_lineup_stint["duration_seconds"].sum())

    f = (
        fact_lineup_stint
        .sort_values(["period", "start_seconds_remaining"], ascending=[True, False])
        .reset_index(drop=True)
    )

    boundary_issues = 0

    for i in range(len(f) - 1):
        a = f.iloc[i]
        b = f.iloc[i + 1]

        if int(a["period"]) != int(b["period"]):
            continue

        lineup_changed = (
            set(a["home_lineup_ids"]) != set(b["home_lineup_ids"])
            or set(a["away_lineup_ids"]) != set(b["away_lineup_ids"])
        )

        if lineup_changed and (
            not bool(a["ended_by_substitution"])
            or str(a["end_clock"]) != str(b["start_clock"])
        ):
            boundary_issues += 1

    quality = {
        "game_id": game_id,
        "n_stints": len(fact_lineup_stint),
        "negative_durations": neg_dur,
        "zero_durations": zero_dur,
        "bad_home_lineup_size": bad_home,
        "bad_away_lineup_size": bad_away,
        "total_seconds": total_sec,
        "boundary_issues": boundary_issues,
        "status": "success"
    }

    quality_df = pd.DataFrame([quality])

    return fact_lineup_stint, quality_df, lineup_debug_df


# -----------------------------
# Run 5-game test
# -----------------------------
game_ids = get_season_game_ids(
    season=SEASON,
    season_type=SEASON_TYPE,
    n_games=N_GAMES
)

print("Testing these game IDs:")
print(game_ids)

all_quality = []
errors = []

for i, game_id in enumerate(game_ids, start=1):
    print(f"\n[{i}/{len(game_ids)}] Processing game {game_id}")

    try:
        stints_df, quality_df, debug_df = build_lineup_stints_for_game(game_id)

        # Local paths
        stint_local_path = (
            LOCAL_OUTPUT_DIR
            / f"season={SEASON}"
            / f"game_id={game_id}"
            / "stints.parquet"
        )

        quality_local_path = (
            LOCAL_OUTPUT_DIR
            / f"season={SEASON}"
            / f"game_id={game_id}"
            / "quality.parquet"
        )

        debug_local_path = (
            LOCAL_OUTPUT_DIR
            / f"season={SEASON}"
            / f"game_id={game_id}"
            / "period_start_debug.parquet"
        )

        # S3 keys
        stint_s3_key = f"{S3_PREFIX}/season={SEASON}/game_id={game_id}/stints.parquet"
        quality_s3_key = f"{S3_PREFIX}/season={SEASON}/game_id={game_id}/quality.parquet"
        debug_s3_key = f"{S3_PREFIX}/season={SEASON}/game_id={game_id}/period_start_debug.parquet"

        stint_path = save_df_local_and_s3(
            stints_df,
            local_path=stint_local_path,
            s3_bucket=S3_BUCKET,
            s3_key=stint_s3_key,
            upload=S3_UPLOAD
        )

        quality_path = save_df_local_and_s3(
            quality_df,
            local_path=quality_local_path,
            s3_bucket=S3_BUCKET,
            s3_key=quality_s3_key,
            upload=S3_UPLOAD
        )

        debug_path = save_df_local_and_s3(
            debug_df,
            local_path=debug_local_path,
            s3_bucket=S3_BUCKET,
            s3_key=debug_s3_key,
            upload=S3_UPLOAD
        )

        all_quality.append(quality_df)

        print(f"Saved stints: {stint_path}")
        print(f"Saved quality: {quality_path}")
        print(quality_df.to_string(index=False))

    except Exception as e:
        print(f"FAILED game {game_id}: {e}")

        errors.append({
            "game_id": game_id,
            "error": str(e),
            "traceback": traceback.format_exc()
        })

    # Avoid hammering NBA API
    time.sleep(1.5)


# -----------------------------
# Summary
# -----------------------------
quality_summary_df = pd.concat(all_quality, ignore_index=True) if all_quality else pd.DataFrame()
error_df = pd.DataFrame(errors)

summary_dir = LOCAL_OUTPUT_DIR / f"season={SEASON}" / "_summary"
summary_dir.mkdir(parents=True, exist_ok=True)

quality_summary_df.to_csv(summary_dir / "quality_summary.csv", index=False)
error_df.to_csv(summary_dir / "errors.csv", index=False)

print("\nDONE")
print("Successful games:", len(quality_summary_df))
print("Failed games:", len(error_df))

display(quality_summary_df)

if not error_df.empty:
    display(error_df[["game_id", "error"]])

Testing these game IDs:
['0022400062', '0022400061', '0022400064', '0022400069', '0022400070']

[1/5] Processing game 0022400062
FAILED game 0022400062: [Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400062.json'

[2/5] Processing game 0022400061
FAILED game 0022400061: [Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400061.json'

[3/5] Processing game 0022400064
FAILED game 0022400064: [Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400064.json'

[4/5] Processing game 0022400069
FAILED game 0022400069: [Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400069.json'

[5/5] Processing game 0022400070
FAILED game 0022400070: [Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400070.json'


""


,game_id,error
0,0022400062,[Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400062...
1,0022400061,[Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400061...
2,0022400064,[Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400064...
3,0022400069,[Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400069...
4,0022400070,[Errno 2] No such file or directory: '/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400070...
